# Landsat 8 Land Surface Temperature (LST)

**Preferred batch path:**

```bash
python transformation/landsat_lst/extract_landsat_lst.py --site plymouth
```

Drive shard merge (only if `GEE_EXPORT_MODE=drive` produced tiles):

```bash
python transformation/landsat_lst/extract_landsat_lst.py --site plymouth --merge-shards
```

This notebook remains useful for interactive maps / QA. Exports land under `heat_hazard/sites/<city>/data/input/`.

**Dataset:** USGS Landsat 8 Level-2, Collection 2, Tier 1 (`LANDSAT/LC08/C02/T1_L2`)  
**Period:** 2015–2024, austral summer only (DJF — December, January, February)  
**Product:** P90 composite of cloud-free surface temperature → heat hazard layer

## Methodology

### Why DJF (austral summer)?
Porto Alegre (~30°S) experiences its hottest temperatures in December–February.
Restricting to DJF captures **peak thermal stress** and maximises the UHI signal;
off-season scenes would dilute the hazard.

### Why P90?
- The 90th-percentile temperature over 10 summers represents a **chronic high-heat condition**, not a single anomalous event.
- It is more robust to remaining cloud artefacts than the maximum, and more representative of sustained exposure than the median.
- Consistent with literature on heat vulnerability mapping (e.g. Heaviside et al. 2017).

### Processing steps
1. Filter collection to DJF 2015–2024, POA bounds, `PROCESSING_LEVEL == 'L2SP'`, cloud cover < 30 %.
2. Apply official scaling factors: `LST_K = ST_B10 × 0.00341802 + 149.0`; convert to °C.
3. Pixel-level cloud / shadow mask using `QA_PIXEL` bits (1 dilated cloud, 3 cloud, 4 shadow).
4. Reduce collection to P90 per pixel → `lst_p90_celsius`.
5. Count valid (unmasked) observations per pixel → `obs_count` (QA layer).
6. Compute LST anomaly relative to the POA mean → `lst_anomaly_celsius`.
7. Min-max normalization within POA → `lst_norm` (0–1, input for heat hazard ensemble).

### Outputs
| File | Description |
|------|-------------|
| `lst_lc08_p90_djf_2015_2024_poa.tif` | P90 LST in °C, 30 m native resolution |
| `lst_lc08_obs_count_djf_2015_2024_poa.tif` | Valid observation count per pixel |
| `lst_lc08_norm_djf_2015_2024_poa.tif` | Min-max normalized P90 (0–1) within POA |

In [1]:
# Site configuration — uses transformation/heat_hazard city configs
import os
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_HEAT_HAZARD = None
for _candidate in [_HERE, *_HERE.parents]:
    _probe = _candidate / "heat_hazard" if _candidate.name != "heat_hazard" else _candidate
    if (_probe / "site_config.py").is_file() and (_probe / "config" / "sites").is_dir():
        _HEAT_HAZARD = _probe
        break
if _HEAT_HAZARD is None:
    raise FileNotFoundError("Could not locate transformation/heat_hazard from notebook cwd")

sys.path.insert(0, str(_HEAT_HAZARD))
from site_config import load_site_config

HEAT_HAZARD_ROOT = _HEAT_HAZARD
HEAT_ROOT = HEAT_HAZARD_ROOT  # backward-compatible alias

# Set the city here (edit this line). That value wins for interactive runs.
# Use None to fall back to env HEAT_SITE (default porto_alegre).
SITE_SLUG = "plymouth"  # or: "porto_alegre" | "edina" | "richfield" | "rochester" | "apple_valley" | None
if SITE_SLUG is None:
    SITE_SLUG = os.environ.get("HEAT_SITE", "porto_alegre")
SITE_CONFIG = load_site_config(SITE_SLUG, HEAT_HAZARD_ROOT)
SITE_ROOT = SITE_CONFIG["paths_abs"]["site_root"]
INPUT_DIR = SITE_CONFIG["paths_abs"]["data_input"]
INTERMEDIATE_DIR = SITE_CONFIG["paths_abs"]["data_intermediate"]
OUTPUT_DIR = SITE_CONFIG["paths_abs"]["data_output"]
OUT_ROOT = SITE_CONFIG["paths_abs"]["out"]
CACHE_DIR = SITE_CONFIG["paths_abs"]["cache"]
STYLES_DIR = SITE_CONFIG["paths_abs"]["styles"]
OUTPUT_PREFIX = SITE_CONFIG["output_prefix"]
SEASON = SITE_CONFIG["season"]
SEASON_LABEL = SITE_CONFIG["season_label"]
START_YEAR = int(SITE_CONFIG["start_year"])
END_YEAR = int(SITE_CONFIG["end_year"])
print(f"Heat hazard site: {SITE_CONFIG['display_name']} ({SITE_SLUG})")
print(f"Config: {SITE_CONFIG['config_path']}")
print(f"Season: {SEASON_LABEL} {START_YEAR}-{END_YEAR}")
print(f"Inputs -> {INPUT_DIR}")


Heat hazard site: Plymouth (plymouth)
Config: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/config/sites/plymouth.yaml
Season: JJA 2015-2024
Inputs -> /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/data/input


## 0. Setup

In [2]:
import ee
import geemap
import numpy as np

ee.Authenticate()

ee.Initialize(
    project='eecc-maureen',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

## 1. Area of interest (Porto Alegre)

In [3]:
# Site ROI: use the site polygon when available; fall back to bbox.
import json


def load_site_roi() -> ee.Geometry:
    boundary_path = SITE_CONFIG["boundary_path_abs"]
    if boundary_path.exists():
        data = json.loads(boundary_path.read_text())
        if data.get("type") == "FeatureCollection":
            features = [
                ee.Feature(ee.Geometry(feature["geometry"]), feature.get("properties", {}))
                for feature in data.get("features", [])
                if feature.get("geometry")
            ]
            if features:
                return ee.FeatureCollection(features).geometry()
        if data.get("type") == "Feature":
            return ee.Geometry(data["geometry"])
        if data.get("type") in {"Polygon", "MultiPolygon", "GeometryCollection"}:
            return ee.Geometry(data)
    return ee.Geometry.Rectangle(SITE_CONFIG["bbox"])


roi = load_site_roi()
print(f"ROI loaded for {SITE_CONFIG['display_name']} from {SITE_CONFIG['boundary_path_abs']}")


ROI loaded for Plymouth from /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/boundary/site.geojson


## 2. Load and filter Landsat 8 collection

Filters applied:
- Temporal: configured site period (`START_YEAR` → `END_YEAR`)  
- Spatial: configured site ROI  
- Seasonal: configured site season (`SEASON`; e.g. `JJA` = months 6, 7, 8)  
- Processing level: `L2SP` only (both SR and ST bands present)  
- Scene-level cloud cover over land: < 30 %

In [4]:
START_DATE = f'{START_YEAR}-01-01'
END_DATE   = f'{END_YEAR}-12-31'
MAX_CLOUD_LAND = 30           # scene-level cloud % over land

SEASON_MONTHS = {
    'djf': [12, 1, 2],
    'mam': [3, 4, 5],
    'jja': [6, 7, 8],
    'son': [9, 10, 11],
    'annual': list(range(1, 13)),
}
months = SEASON_MONTHS.get(SEASON.lower())

if months is None:
    raise ValueError(f'Unsupported season {SEASON!r}. Expected one of {sorted(SEASON_MONTHS)}')

month_filter = ee.Filter.inList('month', months)

raw = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterDate(START_DATE, END_DATE)
    .filterBounds(roi)
    .map(lambda image: image.set('month', image.date().get('month')))
    .filter(month_filter)
    .filter(ee.Filter.eq('PROCESSING_LEVEL', 'L2SP'))
    .filter(ee.Filter.lte('CLOUD_COVER_LAND', MAX_CLOUD_LAND))
)

print(f'Season months ({SEASON_LABEL}):', months)
print('Scenes in collection (before pixel masking):', raw.size().getInfo())

Season months (JJA): [6, 7, 8]
Scenes in collection (before pixel masking): 22


## 3. Scale factors and cloud mask

**ST_B10 conversion:**
```
LST_K = DN × 0.00341802 + 149.0
LST_°C = LST_K − 273.15
```

**QA_PIXEL bitmask (masked pixels):**
| Bit | Meaning |
|-----|---------|
| 1 | Dilated cloud |
| 3 | Cloud |
| 4 | Cloud shadow |

In [5]:
def apply_scale_factors(image):
    """Apply official Landsat C2 L2 scale factors to ST and SR bands."""
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal = image.select('ST_B.*').multiply(0.00341802).add(149.0)  # Kelvin
    return image.addBands(optical, None, True).addBands(thermal, None, True)


def kelvin_to_celsius(image):
    """Convert ST_B10 from Kelvin to Celsius and rename band."""
    lst_c = image.select('ST_B10').subtract(273.15).rename('lst_celsius')
    return image.addBands(lst_c)


def mask_clouds(image):
    
    """Mask cloud, dilated cloud, and cloud shadow pixels via QA_PIXEL."""
    qa = image.select('QA_PIXEL')
    # Bits 1 (dilated cloud), 3 (cloud), 4 (cloud shadow)
    cloud_bits = (1 << 1) | (1 << 3) | (1 << 4)
    mask = qa.bitwiseAnd(cloud_bits).eq(0)
    return image.updateMask(mask)


def preprocess(image):
    return mask_clouds(kelvin_to_celsius(apply_scale_factors(image)))


collection = raw.map(preprocess)
print('Preprocessed collection size:', collection.size().getInfo())

Preprocessed collection size: 22


## 4. P90 summer composite

Reduce all valid (cloud-free) DJF pixels across 2015–2024 to their 90th percentile.
This captures the **chronic high-heat condition** per pixel, not a single event.

In [6]:
lst_band = collection.select('lst_celsius')

# P90 composite
lst_p90 = lst_band.reduce(ee.Reducer.percentile([90])).rename('lst_p90_celsius')

# Valid observation count per pixel (QA layer)
obs_count = lst_band.reduce(ee.Reducer.count()).rename('obs_count')

# Clip to ROI
lst_p90   = lst_p90.clip(roi)
obs_count = obs_count.clip(roi)

# Quick stats (sample 1000 points)
stats = lst_p90.reduceRegion(
    reducer=ee.Reducer.percentile([5, 25, 50, 75, 90, 95]).combine(
        ee.Reducer.min(), sharedInputs=True
    ).combine(ee.Reducer.max(), sharedInputs=True)
    .combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=roi,
    scale=300,
    maxPixels=1e8,
    bestEffort=True,
).getInfo()

print('LST P90 composite stats (°C):') 
for k, v in sorted(stats.items()):
    print(f'  {k}: {v:.2f}')

LST P90 composite stats (°C):
  lst_p90_celsius_max: 53.66
  lst_p90_celsius_mean: 38.21
  lst_p90_celsius_min: 27.23
  lst_p90_celsius_p25: 35.69
  lst_p90_celsius_p5: 31.45
  lst_p90_celsius_p50: 37.95
  lst_p90_celsius_p75: 40.44
  lst_p90_celsius_p90: 43.82
  lst_p90_celsius_p95: 45.94


## 5. Observation count QA

Pixels with fewer than ~5 valid observations are less reliable.
This cell checks the distribution across POA.

In [7]:
obs_stats = obs_count.reduceRegion(
    reducer=ee.Reducer.percentile([5, 25, 50, 75, 95])
    .combine(ee.Reducer.min(), sharedInputs=True)
    .combine(ee.Reducer.max(), sharedInputs=True),
    geometry=roi,
    scale=300,
    maxPixels=1e8,
    bestEffort=True,
).getInfo()

print('Observation count per pixel (DJF scenes, 2015-2024):')
for k, v in sorted(obs_stats.items()):
    print(f'  {k}: {v:.1f}')

Observation count per pixel (DJF scenes, 2015-2024):
  obs_count_max: 22.0
  obs_count_min: 15.0
  obs_count_p25: 18.0
  obs_count_p5: 18.0
  obs_count_p50: 19.0
  obs_count_p75: 20.0
  obs_count_p95: 21.0


## 6. LST anomaly and min-max normalization

**LST anomaly** = P90 LST − mean(P90 LST over POA)  
→ captures how much hotter each pixel is relative to the city average (UHI signal).  
→ Positive values = urban heat hotspots; negative = parks, water, cool corridors.

**Normalization** (0–1 within POA):  
```
lst_norm = (lst_p90 − min_POA) / (max_POA − min_POA)
```
This is the layer used as input to the heat hazard ensemble.

In [8]:
# Compute POA mean and min/max for normalization at 300m scale
roi_stats = lst_p90.reduceRegion(
    reducer=ee.Reducer.mean()
    .combine(ee.Reducer.min(), sharedInputs=True)
    .combine(ee.Reducer.max(), sharedInputs=True),
    geometry=roi,
    scale=300,
    maxPixels=1e8,
    bestEffort=True,
)

poa_mean = ee.Number(roi_stats.get('lst_p90_celsius_mean'))
poa_min  = ee.Number(roi_stats.get('lst_p90_celsius_min'))
poa_max  = ee.Number(roi_stats.get('lst_p90_celsius_max'))

# LST anomaly (°C relative to city mean)
lst_anomaly = lst_p90.subtract(poa_mean).rename('lst_anomaly_celsius')

# Min-max normalization [0, 1]
lst_norm = (
    lst_p90.subtract(poa_min)
    .divide(poa_max.subtract(poa_min))
    .rename('lst_norm')
)

print('POA stats:')
print('  Mean P90 LST (°C):', round(poa_mean.getInfo(), 2))
print('  Min  P90 LST (°C):', round(poa_min.getInfo(), 2))
print('  Max  P90 LST (°C):', round(poa_max.getInfo(), 2))

POA stats:
  Mean P90 LST (°C): 38.21
  Min  P90 LST (°C): 27.23
  Max  P90 LST (°C): 53.66


In [9]:
## Summary statistics for all three output bands

layers = [
    ('LST P90 (°C)',         lst_p90,    'lst_p90_celsius'),
    ('LST Anomaly (°C)',     lst_anomaly,'lst_anomaly_celsius'),
    ('LST Normalized (0-1)', lst_norm,   'lst_norm'),
]

print(f"{'Layer':<26} {'Min':>8} {'Mean':>8} {'Max':>8}")
print("-" * 54)
for label, img, band in layers:
    s = img.reduceRegion(
        reducer=ee.Reducer.min()
            .combine(ee.Reducer.mean(), sharedInputs=True)
            .combine(ee.Reducer.max(),  sharedInputs=True),
        geometry=roi,
        scale=300,
        maxPixels=1e8,
        bestEffort=True,
    ).getInfo()
    vmin  = s[f'{band}_min']
    vmean = s[f'{band}_mean']
    vmax  = s[f'{band}_max']
    print(f"{label:<26} {vmin:>8.3f} {vmean:>8.3f} {vmax:>8.3f}")

Layer                           Min     Mean      Max
------------------------------------------------------
LST P90 (°C)                 27.231   38.210   53.655
LST Anomaly (°C)            -10.979   -0.000   15.445
LST Normalized (0-1)          0.000    0.415    1.000


## 7. Visualization

In [10]:
# Visualization parameters
vis_lst = {
    'min': 20, 'max': 50,
    'palette': ['#313695','#4575b4','#74add1','#abd9e9',
                '#fee090','#fdae61','#f46d43','#d73027','#a50026'],
}
vis_anomaly = {
    'min': -15, 'max': 15,
    'palette': ['#2166ac','#92c5de','#f7f7f7','#f4a582','#b2182b'],
}
vis_norm = {
    'min': 0, 'max': 1,
    'palette': ['#440154','#31688e','#35b779','#6ece58','#fde725'],  # viridis
}
vis_obs = {
    'min': 0, 'max': 30,
    'palette': ['#d7191c','#fdae61','#ffffbf','#a6d96a','#1a9641'],
}

m = geemap.Map(center=[46.28, -94.30], zoom=11)
#m = geemap.Map(center=[-30.10, -51.16], zoom=11)
m.add_basemap('OpenStreetMap.Mapnik')

m.addLayer(lst_p90,    vis_lst,     'LST P90 (°C)',         True)
m.addLayer(lst_anomaly, vis_anomaly, 'LST Anomaly (°C)',     False)
m.addLayer(lst_norm,   vis_norm,    'LST Normalized (0-1)', False)
m.addLayer(obs_count,  vis_obs,     'Obs Count',            False)

m.add_colorbar(
    vis_params=vis_lst,
    label='LST P90 (°C)',
    orientation='horizontal',
    position='bottomright',
    transparent_bg=True,
)
m.add_layer_control()
m

Map(center=[46.28, -94.3], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

## 8. Export to Google Drive

Three GeoTIFFs exported at **30 m** (Landsat native resolution):

| Task | File | Description |
|------|------|-------------|
| `task_lst_p90` | `lst_lc08_p90_djf_2015_2024_poa` | P90 LST in °C |
| `task_obs_count` | `lst_lc08_obs_count_djf_2015_2024_poa` | Valid observation count |
| `task_lst_norm` | `lst_lc08_norm_djf_2015_2024_poa` | Min-max normalized P90 (0–1) |

**Note:** Exports go to `EE_exports/heat/` in your Google Drive. Run `.start()` and monitor via the [GEE Task Manager](https://code.earthengine.google.com/tasks).

In [11]:
# --- Export Landsat LST layers to heat site input/ (local by default) ---
# Override: export GEE_EXPORT_MODE=drive

from gee_local_export import export_image_to_input

EXPORT_SCALE = 30
EXPORT_CRS = "EPSG:4326"
EXPORT_FOLDER = "EE_exports/heat"

site_label = SITE_CONFIG["display_name"]
period_label = f"{SEASON_LABEL} {START_YEAR}-{END_YEAR}"
layer_names = SITE_CONFIG["layers"]

export_cfg = [
    (
        lst_p90,
        layer_names["landsat_p90"],
        f"P90 LST {period_label} {site_label} (degC, Landsat 8 C2 L2)",
    ),
    (
        obs_count,
        layer_names["landsat_obs_count"],
        f"Valid observation count {period_label} {site_label} (Landsat 8 C2 L2)",
    ),
    (
        lst_norm,
        layer_names["landsat_norm"],
        f"Min-max normalized P90 LST {period_label} {site_label} (0-1)",
    ),
]

for image, filename, desc in export_cfg:
    export_image_to_input(
        image,
        filename=filename,
        region=roi,
        scale=EXPORT_SCALE,
        input_dir=INPUT_DIR,
        crs=EXPORT_CRS,
        description=Path(filename).stem,
        drive_folder=EXPORT_FOLDER,
    )

print("Exports complete. Files land under:", INPUT_DIR)


[local] exporting lst_lc08_p90_jja_2015_2024_plymouth.tif → /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/data/input/lst_lc08_p90_jja_2015_2024_plymouth.tif (scale=30m, crs=EPSG:4326)
Generating URL ...
Please wait ...
Data downloaded to /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/data/input/lst_lc08_p90_jja_2015_2024_plymouth.tif
[local] wrote /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/data/input/lst_lc08_p90_jja_2015_2024_plymouth.tif (0.59 MB)
[local] exporting lst_lc08_obs_count_jja_2015_2024_plymouth.tif → /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/data/input/lst_lc08_obs_count_jja_2015_2024_plymouth.tif (scale=30m, crs=EPSG:4326)
Generating URL ...
Please wait ...
Data downloaded to /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/data/input/lst_lc08_obs_count_jja_2015_2024_plymouth.tif
[local

## 9. Check export status

In [ ]:
# Run this cell to check task status without reloading the page
for name, task in tasks:
    status = task.status()
    state  = status.get('state', 'UNKNOWN')
    error  = status.get('error_message', '')
    print(f'{name}: {state}' + (f' — {error}' if error else ''))

## 10. Merge GEE export shards

Large ROIs (e.g. Minnesota at 30 m) are split by Earth Engine into multiple GeoTIFFs:

`{name}-0000000000-0000000000.tif`, `{name}-0000000000-0000023296.tif`, …

This cell mosaics those shards into the single filenames expected by `SITE_CONFIG["layers"]` under `sites/<site>/data/input/`.

In [ ]:
# Merge Earth Engine Drive shards into single GeoTIFFs expected by site config.
from pathlib import Path
import os
import re
import shutil
import subprocess
import tempfile

INPUT_DIR.mkdir(parents=True, exist_ok=True)

layer_keys = ("landsat_p90", "landsat_norm")
shard_re = re.compile(r"^.+-\d{10}-\d{10}\.tif$", re.IGNORECASE)

# Jupyter kernels often lack Homebrew on PATH.
GDAL_CANDIDATE_DIRS = [
    Path("/opt/homebrew/bin"),
    Path("/usr/local/bin"),
    Path(os.environ.get("HOME", "")) / "homebrew" / "bin",
]


def resolve_gdal_tool(name: str) -> str:
    found = shutil.which(name)
    if found:
        return found
    for directory in GDAL_CANDIDATE_DIRS:
        candidate = directory / name
        if candidate.is_file() and os.access(candidate, os.X_OK):
            return str(candidate)
    raise FileNotFoundError(
        f"Could not find `{name}`. Install GDAL (e.g. `brew install gdal`) "
        f"or add it to PATH. Checked: PATH plus {', '.join(str(p) for p in GDAL_CANDIDATE_DIRS)}"
    )


GDALBUILDVRT = resolve_gdal_tool("gdalbuildvrt")
GDAL_TRANSLATE = resolve_gdal_tool("gdal_translate")
print(f"Using gdalbuildvrt: {GDALBUILDVRT}")
print(f"Using gdal_translate: {GDAL_TRANSLATE}")


def find_shards(stem: str) -> list[Path]:
    return sorted(
        p for p in INPUT_DIR.glob(f"{stem}-*.tif")
        if shard_re.match(p.name)
    )


def merge_shards(stem: str) -> Path:
    out_tif = INPUT_DIR / f"{stem}.tif"
    shards = find_shards(stem)

    if out_tif.exists() and not shards:
        print(f"Already merged (no shards found): {out_tif}")
        return out_tif
    if not shards:
        raise FileNotFoundError(
            f"No GEE shards found for {stem!r} in {INPUT_DIR}. "
            f"Expected files like {stem}-0000000000-0000000000.tif"
        )

    print(f"Merging {len(shards)} shards → {out_tif.name}")
    for shard in shards:
        print(f"  - {shard.name} ({shard.stat().st_size / 1e9:.2f} GB)")

    with tempfile.TemporaryDirectory(prefix=f"{stem}_vrt_") as tmp:
        vrt_path = Path(tmp) / f"{stem}.vrt"
        subprocess.run(
            [GDALBUILDVRT, str(vrt_path), *[str(p) for p in shards]],
            check=True,
        )
        # Write to a temp path in INPUT_DIR, then replace, so a failed run
        # does not leave a half-written final file.
        tmp_out = INPUT_DIR / f".{stem}.merging.tif"
        if tmp_out.exists():
            tmp_out.unlink()
        subprocess.run(
            [
                GDAL_TRANSLATE,
                str(vrt_path),
                str(tmp_out),
                "-of", "GTiff",
                "-co", "COMPRESS=LZW",
                "-co", "TILED=YES",
                "-co", "BIGTIFF=YES",
                "-co", "NUM_THREADS=ALL_CPUS",
            ],
            check=True,
        )
        tmp_out.replace(out_tif)

    print(f"Wrote: {out_tif} ({out_tif.stat().st_size / 1e9:.2f} GB)\n")
    return out_tif


merged = {}
for key in layer_keys:
    stem = SITE_CONFIG["layers"][key].removesuffix(".tif")
    merged[key] = merge_shards(stem)

print("Done. Merged layers:")
for key, path in merged.items():
    print(f"  {key}: {path}")


## 10. Convert LST P90 to COG and Generate Tiles

Publish the local `sites/<site_slug>/data/input/lst_lc08_p90_djf_2015_2024_poa.tif` raster for web maps: COG + colorized XYZ tiles + value-encoded XYZ tiles for hover lookup.

Requires GDAL CLI (`gdal_translate`, `gdaldem`, `gdal_calc.py`, `gdal2tiles.py`) and `sites/<site_slug>/data/output/lst_lc08_p90_djf_2015_2024_poa_colors.txt`.

In [12]:
# Convert Landsat 8 LST P90 GeoTIFF to COG + visual tiles + value tiles.
from pathlib import Path
import shutil
import subprocess



PROJECT_ROOT = HEAT_HAZARD_ROOT
in_tif = INPUT_DIR / SITE_CONFIG["layers"]["landsat_p90"]
slug_base = SITE_CONFIG["layers"]["landsat_p90"].removesuffix(".tif")
out_dir = OUT_ROOT / slug_base
colors_txt = STYLES_DIR / f"{slug_base}_colors.txt"

slug = SITE_CONFIG["layers"]["landsat_p90"].removesuffix(".tif")
cog_tif = out_dir / f"{slug}_cog.tif"
colorized_tif = out_dir / f"{slug}_colorized.tif"
value_encoded_tif = out_dir / f"{slug}_value_encoded_rgb.tif"
tiles_dir = out_dir / "tiles_visual"
value_tiles_dir = out_dir / "tiles_values"
decode_txt = out_dir / f"{slug}_value_tiles_decode.txt"

out_dir.mkdir(parents=True, exist_ok=True)
if not in_tif.exists():
    raise FileNotFoundError(f"Missing input raster: {in_tif}")
if not colors_txt.exists():
    raise FileNotFoundError(f"Missing color table: {colors_txt}")

print("Input:", in_tif)
print("Output dir:", out_dir)

# 1) COG preserving raw LST values in degrees Celsius.
subprocess.run([
    "gdal_translate", str(in_tif), str(cog_tif),
    "-of", "COG",
    "-ot", "Float32",
    "-co", "COMPRESS=DEFLATE",
    "-co", "RESAMPLING=NEAREST",
    "-co", "OVERVIEWS=AUTO",
], check=True)
print("Created COG:", cog_tif)

# 2) Colorized raster + visual XYZ tiles.
subprocess.run([
    "gdaldem", "color-relief",
    str(cog_tif), str(colors_txt), str(colorized_tif),
    "-alpha",
], check=True)
print("Created colorized raster:", colorized_tif)

# gdal2tiles preflight: use the Python interpreter referenced by gdal2tiles.py.
gdal2tiles = shutil.which("gdal2tiles.py")
if not gdal2tiles:
    raise RuntimeError("gdal2tiles.py not found in PATH. Install GDAL (e.g. brew install gdal).")

gdal2tiles_python = None
with open(gdal2tiles, "r", encoding="utf-8", errors="ignore") as f:
    first_line = f.readline().strip()
if first_line.startswith("#!"):
    shebang_parts = first_line[2:].split()
    if shebang_parts:
        if shebang_parts[0].endswith("env") and len(shebang_parts) > 1:
            gdal2tiles_python = shutil.which(shebang_parts[1])
        else:
            gdal2tiles_python = shebang_parts[0]
if not gdal2tiles_python:
    gdal2tiles_python = shutil.which("python3") or shutil.which("python")
subprocess.run([gdal2tiles_python, "-c", "import numpy"], check=True, capture_output=True)

tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py",
    "-r", "near",
    "-z", "8-15",
    "--xyz",
    "-w", "none",
    str(colorized_tif),
    str(tiles_dir),
], check=True)
print("Visual tiles:", tiles_dir)

# 3) Value tiles for client-side hover.
# Encodes Celsius at 0.01 C precision with +100 C offset plus 1 so encoded RGB value 0 can mean nodata.
# Decode: encoded = R + 256*G + 65536*B; lst_celsius = ((encoded - 1) / 100) - 100; encoded == 0 => nodata.
base_expr = (
    "numpy.where(numpy.isnan(A), 0, "
    "numpy.rint(numpy.clip(A + 100,0,167772.14)*100).astype(numpy.int64) + 1)"
)
subprocess.run([
    "gdal_calc.py",
    "-A", str(cog_tif),
    "--calc", f"bitwise_and({base_expr},255)",
    "--calc", f"bitwise_and(right_shift({base_expr},8),255)",
    "--calc", f"bitwise_and(right_shift({base_expr},16),255)",
    "--type", "Byte",
    "--NoDataValue", "0",
    "--overwrite",
    "--outfile", str(value_encoded_tif),
], check=True)

value_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py",
    "-r", "near",
    "-z", "8-15",
    "--xyz",
    "-w", "none",
    str(value_encoded_tif),
    str(value_tiles_dir),
], check=True)

metadata = """Landsat 8 LST P90 DJF 2015-2024 value tiles

Source raster: sites/<site_slug>/data/input/lst_lc08_p90_djf_2015_2024_poa.tif
COG: sites/<site_slug>/data/output/lst_lc08_p90_djf_2015_2024_poa/lst_lc08_p90_djf_2015_2024_poa_cog.tif
Visual tiles: sites/<site_slug>/data/output/lst_lc08_p90_djf_2015_2024_poa/tiles_visual/{z}/{x}/{y}.png
Value tiles: sites/<site_slug>/data/output/lst_lc08_p90_djf_2015_2024_poa/tiles_values/{z}/{x}/{y}.png

Value tile encoding:
encoded = R + 256 * G + 65536 * B
if encoded == 0: value is nodata
else: lst_celsius = ((encoded - 1) / 100) - 100

LST values are P90 land surface temperature in degrees Celsius, encoded at 0.01 C precision.
"""
decode_txt.write_text(metadata, encoding="utf-8")

print("Value tiles:", value_tiles_dir)
print("Decode metadata:", decode_txt)
print("Decode: encoded = R + 256*G + 65536*B; lst_celsius = ((encoded - 1) / 100) - 100; encoded 0 = nodata")


FileNotFoundError: Missing color table: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/styles/lst_lc08_p90_jja_2015_2024_plymouth_colors.txt